In [ ]:
!pip install transformers
!pip install "transformers[torch]"

In [ ]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

: 

In [ ]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")


: 

In [ ]:
train_data.sample(10)

: 

In [1]:
# Random sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

NameError: name 'train_data' is not defined

In [ ]:
train_data.shape

## Data pre- processing

In [1]:
import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text) #remove lines
    text = re.sub(r"\s+", " ",text) # remove spaces
    text = re.sub(r"<.*?>", " ",text) # remove HTML tags <p> <h>
    text = text.strip().lower()
    return text
    

In [2]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

NameError: name 'train_data' is not defined

In [ ]:
train_data["dialogue"][0]

### Tokenize

In [3]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

NameError: name 'T5Tokenizer' is not defined

In [ ]:
# raw data =. tokenized inputs for fine-tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length = 512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length = 150, truncation=True)

    inputs["labels"] = targets["input_ids"] #token ids  => add to input as labels
    return inputs

In [ ]:
train_dataset = train_data.apply(tokenize, axis=1).tolist() # convert to list bcz list is compatible with huggingface
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [ ]:
type(train_dataset[0]["input_ids"])
type(train_dataset[0]["input_ids"])

### Working with our model

In [ ]:
# NLP => generation task
# T5ForConditionalGeneration for conditional genration task used in this project to get summary from given texts

model = T5ForConditionalGeneration.from_pretrained("t5-small")

In [4]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")      # Apple Silicon GPU
elif torch.cuda.is_available():
    device = torch.device("cuda")     # NVIDIA GPU
else:
    device = torch.device("cpu")      # CPU

print("Device:", device)

model.to(device)

ModuleNotFoundError: No module named 'torch'

In [5]:
# Training Arguments

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs = 6,
    weight_decay = 0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps = 500,
    # 0 => lr default
)

NameError: name 'TrainingArguments' is not defined

### Trainer

In [6]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

NameError: name 'Trainer' is not defined

### Save the model

In [7]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

NameError: name 'model' is not defined

### Test the core logic for summarization

In [8]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # clean

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding = "max_length",
        max_length = 512,
        truncation = True,
        return_tensors = "pt"
    )

    
    inputs = {key: value.to(device) for key, value in inputs.items()} # our model and data should on same device

    # generate the summary => token ids
    targets = model.generate(
        input_ids  = inputs["input_ids"], #token ids
        attention_mask = inputs["attention_mask"], # show amount of paddings
        max_length = 150,
        num_beams = 4, #transformers gives 4 different summaries and will give the best among them
        early_stopping = True, # 
        
    )

    #decode our output
    # token ids  converts to text summary => decoding 
    summary = tokenizer.decode(targets[0], skip_special_tokens = True) # EOS , SEP
    return summary

In [ ]:
test_dialogue = """"
Reporter: Good morning, Dr. Sharma. Thank you for joining us today. To begin, could you explain what Artificial Intelligence really means?

AI/ML Scientist: Good morning. Artificial Intelligence, or AI, is a field of computer science that focuses on creating machines and software capable of performing tasks that normally require human intelligence. These tasks include understanding language, recognizing images, making decisions, solving problems, and learning from experience.

Reporter: Many people use the terms AI and Machine Learning together. Are they the same thing?

AI/ML Scientist: Not exactly. AI is the broader concept. Machine Learning, or ML, is a subfield of AI. Instead of explicitly programming a computer with every rule, we give it data and allow it to learn patterns from that data.

Reporter: Could you give us a simple example?

AI/ML Scientist: Of course. Imagine we want a computer to recognize cats and dogs. Instead of writing thousands of rules about ears, fur, and noses, we give the computer thousands of labeled images of cats and dogs. A machine learning algorithm studies these examples and learns patterns that help it classify new images.

Reporter: So, is machine learning similar to the way humans learn?

AI/ML Scientist: In some ways, yes. Humans learn from experience and examples. Similarly, machine learning models learn patterns from data. However, machines do not understand the world exactly as humans do. They use mathematical patterns and statistical relationships.

Reporter: What are the major types of machine learning?

AI/ML Scientist: The three major types are supervised learning, unsupervised learning, and reinforcement learning.

In supervised learning, the model learns from labeled data. For example, we might give it emails labeled as "spam" or "not spam."

In unsupervised learning, the data does not have labels. The model tries to discover hidden patterns or groups in the data.

In reinforcement learning, an agent learns by interacting with an environment. It receives rewards for good actions and penalties for bad actions. This is commonly used in robotics and game-playing systems.

Reporter: We hear a lot about Generative AI today. What exactly is it?

AI/ML Scientist: Generative AI is a type of AI that can create new content. It can generate text, images, music, video, and computer code. Large Language Models, or LLMs, are examples of Generative AI systems that can understand and generate human-like text.

Reporter: How do systems like chatbots learn to generate text?

AI/ML Scientist: They are trained on very large collections of text. During training, the model learns relationships between words and concepts. For example, it learns that certain words commonly appear together and that language follows certain patterns. Modern models use neural networks, particularly Transformer architectures, to process and generate language.

Reporter: Does that mean AI actually understands everything it says?

AI/ML Scientist: This is an important question. AI systems can process information and generate very convincing responses, but their "understanding" is different from human understanding. They can recognize patterns and relationships, but they do not necessarily possess human consciousness, emotions, or personal experiences.

Reporter: What are some important applications of AI and ML today?

AI/ML Scientist: AI and ML are being used in healthcare, finance, education, transportation, agriculture, cybersecurity, robotics, and entertainment. For example, AI can help doctors analyze medical images, help banks detect fraud, personalize education, predict crop diseases, and assist autonomous vehicles.

Reporter: AI sounds extremely powerful. Are there also risks?

AI/ML Scientist: Absolutely. AI can produce incorrect information, inherit biases from its training data, affect privacy, and potentially impact employment. There are also concerns about the misuse of AI. Therefore, responsible AI development is very important.

Reporter: What does responsible AI mean?

AI/ML Scientist: It means developing AI systems that are fair, transparent, secure, reliable, and respectful of privacy. Humans must remain responsible for how powerful AI systems are designed and used.

Reporter: Many students want to enter the AI/ML field. What should they learn first?

AI/ML Scientist: I recommend starting with mathematics, Python programming, data structures, and basic statistics. Then students can learn libraries such as NumPy, Pandas, Scikit-learn, and PyTorch. After learning the fundamentals, they should build practical projects.

Reporter: What kind of projects would you recommend for beginners?

AI/ML Scientist: Beginners could build a house-price prediction model, a spam email classifier, an image classifier, a recommendation system, or a simple reinforcement learning agent. Projects help students understand how theory works in the real world.

Reporter: Finally, what do you think the future of AI and Machine Learning will look like?

AI/ML Scientist: AI will become an increasingly important tool in many areas of life. I believe the future will not simply be about humans versus machines. Instead, the most successful future will involve humans and AI working together. AI can help us analyze large amounts of information and automate repetitive tasks, while humans provide creativity, judgment, empathy, and ethical responsibility.

Reporter: That is a very interesting perspective. Thank you, Dr. Sharma, for explaining AI and Machine Learning in such a clear way.

AI/ML Scientist: Thank you for having me. It was a pleasure discussing this important topic.

Reporter: And that concludes our interview. AI and Machine Learning are transforming the world, but understanding their benefits and responsibilities will be essential as we move toward the future.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary:", summary)